# Sentinel-1 ARD processing

This notebook runs the Sentinel-1 ARD (Analysis Ready Data) processing.

For now, it:
1. Stages the input products from the [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/) (CDSE)
   into the RS-Python catalog: three IW SLC products in the legacy SAFE format (`s1-l1-iw-slc` collection) and the
   three IW ETAD products of the same acquisitions (`s1-l1-iw-etad` collection).
2. Converts the SAFE SLC products to the Zarr format of the EOPF processors with the `convert-safe` Prefect flow, and
   stores them in the `s01siwslc` collection.
3. Runs the S1 ARD reference pipeline on a master SLC in Zarr format, to generate the `REFERENCE_DB`.

It uses the `copernicus` account, not your Jupyter login: the catalog collections and the staged and converted
products belong to `copernicus`, and the flows run as `copernicus`.

## 1. Initialization

In [ ]:
# Owner of the catalog collections and of the staging flows (not your Jupyter login)
OWNER = "copernicus"

# Init environment before running a demo notebook.
from resources.utils import *

init_demo(owner_id=OWNER)

# Reload the global vars again
from resources.utils import *

from datetime import timedelta, timezone

from prefect.client.orchestration import get_client
from prefect.deployments import arun_deployment

print(f"Prefect dashboard public URL: {os.environ['RSPY_PREFECT_URL']}/dashboard")

## 2. Input products

The input products are read from the CDSE STAC catalog. For each one, we give the assets to stage and the RS-Python
catalog collection where it is staged.

In [ ]:
# CDSE STAC API
CDSE_STAC_URL = "https://stac.dataspace.copernicus.eu/v1"

# RS-Python catalog collections where the input products are staged from the CDSE
LEGACY_SLC_COLLECTION = "s1-l1-iw-slc"  # SLC in the legacy SAFE format
ETAD_COLLECTION = "s1-l1-iw-etad"

# RS-Python catalog collection of the SLC products converted to Zarr, read by the S1 ARD processor
SLC_COLLECTION = "s01siwslc"
SLC_PRODUCT_TYPE = "S01SIWSLC"


def zarr_item_id(safe_item_id: str) -> str:
    """Return the catalog item ID of a SAFE product converted to Zarr by the convert-safe flow."""
    # The converted product is named after the SAFE product, with the .zarr extension that is kept in the
    # item ID because it is not an EOPF product name (starting with S0)
    return f"{safe_item_id}.zarr"


# Each input product: (CDSE collection, CDSE item ID, assets to stage, RS-Python catalog collection
# where it is staged).
# The small ETAD products are staged first, then the big SLC products.
input_products = [
    ("sentinel-1-etad", "S1D_IW_ETA__AXDV_20260420T055950_20260420T060023_002426_003FB6_3356", ["product"], ETAD_COLLECTION),
    ("sentinel-1-etad", "S1C_IW_ETA__AXDV_20260408T174632_20260408T174659_007128_00E6F6_727F", ["product"], ETAD_COLLECTION),
    ("sentinel-1-etad", "S1C_IW_ETA__AXDV_20260327T174632_20260327T174659_006953_00E115_4D20", ["product"], ETAD_COLLECTION),
    ("sentinel-1-slc", "S1D_IW_SLC__1SDV_20260420T055950_20260420T060023_002426_003FB6_4AE4", ["product", "thumbnail"], LEGACY_SLC_COLLECTION),
    ("sentinel-1-slc", "S1C_IW_SLC__1SDV_20260408T174632_20260408T174659_007128_00E6F6_E998", ["product", "thumbnail"], LEGACY_SLC_COLLECTION),
    ("sentinel-1-slc", "S1C_IW_SLC__1SDV_20260327T174632_20260327T174659_006953_00E115_D167", ["product", "thumbnail"], LEGACY_SLC_COLLECTION),
]

## 3. Check the Catalog collections

The catalog collections must already exist: this notebook doesn't create them.

In [ ]:
# Raise an error if a collection doesn't exist
legacy_slc_coll = catalog_client.get_collection(LEGACY_SLC_COLLECTION)
etad_coll = catalog_client.get_collection(ETAD_COLLECTION)
slc_coll = catalog_client.get_collection(SLC_COLLECTION)

print(f"S1 IW SLC (SAFE) collection: {legacy_slc_coll.get_links('self')[0].get_href()}")
print(f"S1 IW ETAD collection: {etad_coll.get_links('self')[0].get_href()}")
print(f"S1 IW SLC (Zarr) collection: {slc_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

## 4. Schedule the staging of the input products

The CDSE returns `HTTP 429 Too Many Requests` errors when we download all these products at the same time. So instead
of running the staging immediately with the staging client of `rs_client`, we schedule one run of the `stage-generic`
Prefect flow per product. The first run starts now. The ETAD products are small, so their runs are spaced
by 1 minute only. Then the SLC products are big, so their runs are spaced by 20 minutes.

The products that are already in the catalog are skipped, so you can run this cell again to stage only the missing
products.

In [ ]:
# Prefect deployment of the generic staging flow
STAGING_DEPLOYMENT = "stage-generic/Stage Generic"

# Delay before the start of the next staging run, depending on the size of the staged product
STAGING_INTERVALS = {
    ETAD_COLLECTION: timedelta(minutes=1),
    LEGACY_SLC_COLLECTION: timedelta(minutes=20),
}

scheduled_runs = []
scheduled_time = datetime.now(timezone.utc)
for cdse_collection, item_id, asset_names, catalog_collection in input_products:

    # Skip the products that are already staged
    if list(catalog_client.get_items(catalog_collection, [item_id])):
        print(f"Already staged in {catalog_collection!r}: {item_id}")
        continue

    params = {
        "env": {
            "owner_id": OWNER,
            "called_by": "notebooks/demos/s1_ard_processing.ipynb",
        },
        "stac_input": f"{CDSE_STAC_URL}/collections/{cdse_collection}/items/{item_id}",
        "catalog_collection_identifier": catalog_collection,
        "asset_names": asset_names,
        "poll_interval": 10,
    }

    # Schedule the flow run and return immediately (timeout=0) without waiting for it
    flow_run = await arun_deployment(
        name=STAGING_DEPLOYMENT,
        parameters=params,
        scheduled_time=scheduled_time,
        flow_run_name=f"stage-{item_id}",
        timeout=0,
    )
    scheduled_runs.append(flow_run)
    print(f"Staging of {item_id} scheduled at {scheduled_time:%Y-%m-%d %H:%M:%S %Z}: "
          f"{os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{flow_run.id}")
    scheduled_time += STAGING_INTERVALS[catalog_collection]

## 5. Check the staging runs

Run this cell again to update the state of the staging runs.

In [ ]:
async with get_client() as client:
    for scheduled_run in scheduled_runs:
        flow_run = await client.read_flow_run(scheduled_run.id)
        print(f"{flow_run.name}: {flow_run.state_name} (expected start: {flow_run.expected_start_time:%Y-%m-%d %H:%M:%S %Z})")

## 6. Run the conversion of the SLC products to Zarr

The S1 ARD processor reads the SLC products in the Zarr format of the EOPF processors, not in the legacy SAFE format.
So we convert each SAFE SLC product staged in `s1-l1-iw-slc` with the `convert-safe` Prefect flow, that stores the
Zarr product in the `s01siwslc` collection, with the `<SAFE item ID>.zarr` item ID.

The conversion runs in the CPM v3 Dask cluster of the current user. This notebook connects to it, or creates it if it
doesn't exist, by running `notebooks/init-dask-clusters/init_dask_cluster_cpm3.ipynb`. Keep this notebook open: the
cluster is stopped when its kernel is restarted.

This notebook creates its workers with a single core and 16 GiB of memory, instead of the default resources of the
cpm3 notebook: EOPF loads each measurement TIFF of the SLC product in memory, so a single thread per worker is needed
to convert the biggest IW SLC products. An existing CPM v3 cluster is reused as it is: if yours was created with other
resources, shut it down first from the cpm3 notebook.

We give the `convert-safe` flow the href of the SAFE product item in our catalog, so the flow reads it from
`s1-l1-iw-slc` instead of staging it again: it doesn't download it again from the CDSE. If the staged SAFE product is a
zip archive, the flow replaces it by its extracted content in the object storage and in its `s1-l1-iw-slc` item, which
is kept after the conversion.

The products already converted in the S1 ARD format are skipped. For the others, this cell waits for the end of their
staging runs and starts the conversion of each SLC product as soon as it is staged in `s1-l1-iw-slc`, so you can run
the whole notebook at once. It raises an error if a staging run fails.

The Sentinel-1 SLC products are converted in the format expected by the S1 ARD processor, which runs with CPM
3.0.0rc4: one sub-product per burst and per polarization. This format is NOT compatible with the CPM 3.0.0 SLC model,
which joins both polarizations of each burst since the CPM issue
[#1042](https://gitlab.eopf.copernicus.eu/cpm/eopf-cpm/-/issues/1042): see the
[rs-dpr-service documentation](https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/docs/doc/index.md#sentinel-1-slc-products-s1-ard-format).
The converted items have the `rspy:product_format = s1ard-1.2.0` property.

The SLC items converted before without this property (e.g. in the CPM 3.0.0 format) are rejected by the S1 ARD
processor. This notebook doesn't remove them: the catalog deletes the Zarr product of a removed item from the object
storage with the S3 credentials of the caller, given by OSAM, and the credentials of your Jupyter login cannot delete
the products of `copernicus`. So this cell stops with an error that lists them: remove them from the `s01siwslc`
collection with an API key of `copernicus`, then run this cell again to convert them again. The item must be removed,
not only its Zarr product: the conversion uploads the new Zarr product at the same location without removing the
existing files.

In [ ]:
# Connect to the CPM v3 Dask cluster of the current user, or create it if it doesn't exist,
# by running notebooks/init-dask-clusters/init_dask_cluster_cpm3.ipynb
from resources.dask_clusters.dask_main_env import init_dask_cluster_cpm3

# EOPF reads each measurement TIFF of a S1 SLC SAFE product in a single chunk, which takes ~3.7 GiB in memory for the
# biggest IW SLC products (11 GiB), plus its rechunked copy. A single core gives a single thread per worker, and a
# single thread to the dask scheduler of the conversion subprocess, so only one TIFF is loaded at a time and 16 GiB
# are enough.
cluster_info_cpm3 = await init_dask_cluster_cpm3(worker_cores=1, worker_memory=16)
print(f"CPM v3 Dask cluster: {cluster_info_cpm3.cluster_label!r}")

In [ ]:
import asyncio

# Prefect deployment of the SAFE to Zarr conversion flow
CONVERSION_DEPLOYMENT = "convert-safe/Convert SAFE"

# Format of the SLC products converted for the S1 ARD processor (not compatible with CPM 3.0.0)
S1ARD_PRODUCT_FORMAT = "s1ard-1.2.0"

# Delay between two checks of the staging runs
POLL_INTERVAL = timedelta(minutes=1)

# Staging run of each product scheduled by this notebook
staging_runs = {scheduled_run.name.removeprefix("stage-"): scheduled_run for scheduled_run in scheduled_runs}


async def start_conversion(item_id: str, staged_item):
    """Start the conversion of a staged SAFE SLC product to Zarr and return its flow run, without waiting for it."""
    params = {
        "conversion_input": {
            "env": {
                "owner_id": OWNER,
                "called_by": "notebooks/demos/s1_ard_processing.ipynb",
            },
            # Item of our catalog: the flow converts the SAFE product already staged, without downloading it
            # again from the CDSE
            "stac_input": staged_item.get_self_href(),
            "generated_product_to_collection_identifier": {
                "name": SLC_PRODUCT_TYPE,
                "product_type": SLC_PRODUCT_TYPE,
                "collection_name": SLC_COLLECTION,
            },
            "dask_cluster_label": cluster_info_cpm3.cluster_label,
            "dask_cluster_instance": cluster_info_cpm3.cluster_instance,
        },
    }
    return await arun_deployment(
        name=CONVERSION_DEPLOYMENT,
        parameters=params,
        flow_run_name=f"convert-{item_id}",
        timeout=0,
    )


# SLC products to convert
to_convert = []
# Items converted before in another format, rejected by S1 ARD, that must be removed with an API key of copernicus
old_format_items = []
for _, item_id, _, catalog_collection in input_products:
    if catalog_collection != LEGACY_SLC_COLLECTION:
        continue

    # Skip the products that are already converted in the S1 ARD format
    converted = list(catalog_client.get_items(SLC_COLLECTION, [zarr_item_id(item_id)]))
    if converted and converted[0].properties.get("rspy:product_format") == S1ARD_PRODUCT_FORMAT:
        print(f"Already converted in {SLC_COLLECTION!r}: {zarr_item_id(item_id)}")
        continue
    if converted:
        old_format_items.append(converted[0].id)
    to_convert.append(item_id)

if old_format_items:
    raise RuntimeError(
        f"Converted in another format than {S1ARD_PRODUCT_FORMAT!r}, rejected by S1 ARD: remove these items from "
        f"{OWNER}:{SLC_COLLECTION} with an API key of {OWNER!r}, then run this cell again: {old_format_items}")

# Conversion run of each SLC product, by SAFE item ID
conversion_runs = {}

# Start the conversion of each SLC product as soon as it is staged
async with get_client() as client:
    while to_convert:
        for item_id in list(to_convert):
            staged_items = list(catalog_client.get_items(LEGACY_SLC_COLLECTION, [item_id]))

            # Not staged yet: check the state of its staging run
            if not staged_items:
                if item_id not in staging_runs:
                    raise RuntimeError(
                        f"{item_id!r} is not staged in {LEGACY_SLC_COLLECTION!r} and its staging is not scheduled: "
                        "run the cell that schedules the staging of the input products again")
                flow_run = await client.read_flow_run(staging_runs[item_id].id)
                if not flow_run.state.is_final():
                    continue
                if not flow_run.state.is_completed():
                    raise RuntimeError(
                        f"The staging of {item_id!r} ended in the {flow_run.state_name!r} state: "
                        f"{os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{flow_run.id}")
                # The run is completed: the product is in the catalog now
                staged_items = list(catalog_client.get_items(LEGACY_SLC_COLLECTION, [item_id]))
                if not staged_items:
                    raise RuntimeError(
                        f"The staging of {item_id!r} is completed but the product is not in "
                        f"{LEGACY_SLC_COLLECTION!r}: {os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{flow_run.id}")

            # Staged: start its conversion
            flow_run = await start_conversion(item_id, staged_items[0])
            conversion_runs[item_id] = flow_run
            to_convert.remove(item_id)
            print(f"{datetime.now(timezone.utc):%H:%M:%S} Conversion of {item_id}: "
                  f"{os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{flow_run.id}")

        if to_convert:
            print(f"{datetime.now(timezone.utc):%H:%M:%S} Wait for the staging of {len(to_convert)} SLC product(s)...")
            await asyncio.sleep(POLL_INTERVAL.total_seconds())

## 7. Check the conversion runs

Run this cell again to update the state of the conversion runs.

In [ ]:
async with get_client() as client:
    for conversion_run in conversion_runs.values():
        flow_run = await client.read_flow_run(conversion_run.id)
        print(f"{flow_run.name}: {flow_run.state_name}")

## 8. Run the S1 ARD reference pipeline

Run the `process-s1-ard-reference` Prefect flow on a master SLC, once its conversion to Zarr is finished: the next
cell waits for the end of its conversion run, and raises an error if it fails.
It generates the `REFERENCE_DB` used by the nominal S1 ARD processing.

The processor runs in the S1 ARD Dask cluster of the current user. This notebook connects to it, or creates it if it
doesn't exist, by running `notebooks/init-dask-clusters/init_dask_cluster_s1ard.ipynb`. Keep this notebook open: the
cluster is stopped when its kernel is restarted.

The S1 ARD flows read their settings from the `s1ard-default-setting` Prefect variable. If it doesn't exist, this
notebook creates it with the settings below: processor, Dask cluster, collections of the input SLCs in Zarr format,
auxiliary files and generated products. If it exists, it is not modified, but the flow searches the SLC in its
`input_collections`, so they must contain the `s01siwslc` collection, and must not contain the legacy SAFE SLC
collections (`s1-l1-iw-slc`...): the S1 ARD processor only reads the SLC in Zarr format.

In [ ]:
# Master SLC of the reference pipeline, in SAFE format
REFERENCE_SAFE_SLC = "S1D_IW_SLC__1SDV_20260420T055950_20260420T060023_002426_003FB6_4AE4"

# Master SLC of the reference pipeline, converted to Zarr
REFERENCE_SLC = zarr_item_id(REFERENCE_SAFE_SLC)

# Wait for the end of its conversion run, if it was started by this notebook
if REFERENCE_SAFE_SLC in conversion_runs:
    async with get_client() as client:
        while True:
            flow_run = await client.read_flow_run(conversion_runs[REFERENCE_SAFE_SLC].id)
            if flow_run.state.is_final():
                break
            print(f"{datetime.now(timezone.utc):%H:%M:%S} Wait for the conversion of {REFERENCE_SAFE_SLC} "
                  f"({flow_run.state_name})...")
            await asyncio.sleep(POLL_INTERVAL.total_seconds())
    if not flow_run.state.is_completed():
        raise RuntimeError(
            f"The conversion of {REFERENCE_SAFE_SLC!r} ended in the {flow_run.state_name!r} state: "
            f"{os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{flow_run.id}")

# Check that the SLC is converted in the S1 ARD format
reference_items = list(catalog_client.get_items(SLC_COLLECTION, [REFERENCE_SLC]))
if not reference_items:
    raise RuntimeError(f"{REFERENCE_SLC!r} is not in {SLC_COLLECTION!r}: its conversion to Zarr failed or is not finished")
product_format = reference_items[0].properties.get("rspy:product_format")
if product_format != S1ARD_PRODUCT_FORMAT:
    raise RuntimeError(
        f"{REFERENCE_SLC!r} is not in the {S1ARD_PRODUCT_FORMAT!r} format expected by the S1 ARD processor "
        f"(rspy:product_format = {product_format!r}): the deployed rs-dpr-service must convert the SLC products in "
        "this format (RS-PYTHON/rs-dpr-service#198)")
print(f"{REFERENCE_SLC} is converted in {SLC_COLLECTION!r}, in the {S1ARD_PRODUCT_FORMAT!r} format")

In [ ]:
# Connect to the S1 ARD Dask cluster of the current user (dask-s1ard.<login>.<image tag>), or create it if it doesn't
# exist, by running notebooks/init-dask-clusters/init_dask_cluster_s1ard.ipynb
from resources.dask_clusters.dask_main_env import init_dask_cluster_s1ard

cluster_info_s1ard = await init_dask_cluster_s1ard()
print(f"S1 ARD Dask cluster: {cluster_info_s1ard.cluster_label!r}")

In [ ]:
from prefect.variables import Variable

S1ARD_SETTINGS_VARIABLE = "s1ard-default-setting"

# Settings created if the Prefect variable doesn't exist.
# See https://github.com/RS-PYTHON/rs-client-libraries/blob/develop/docs/doc/advanced/customise-dpr-flow.md#default-settings-of-the-processing-flows
DEFAULT_S1ARD_SETTINGS = {
    "processor": {"name": "s1_ard", "version": "1.2.0"},
    "pipeline": "SLC__1_NRB__1",
    "priority": "low",
    "processing_mode": ["always"],
    "dask_cluster_name": cluster_info_s1ard.cluster_label,
    # SLC converted to Zarr
    "input_collections": ["s01siwslc", "s01sewslc", "s01ssmslc"],
    "auxiliary_product_to_collection_identifier": [
        # ETAD: staged from the CDSE, read from the catalog
        {"product_type": "IW_ETA__AX", "collection_name": "s1-l1-iw-etad", "source": "catalog"},
        {"product_type": "EW_ETA__AX", "collection_name": "s1-l1-ew-etad", "source": "catalog"},
        {"product_type": "SM_ETA__AX", "collection_name": "s1-l1-sm-etad", "source": "catalog"},
        # ORBIT: staged from the AUXIP
        {"product_type": "AUX_POEORB", "collection_name": "s1-aux_poeorb"},
        {"product_type": "MPL_ORBPRE", "collection_name": "s1-mpl_orbpre"},
        {"product_type": "MPL_ORBRES", "collection_name": "s1-mpl_orbres"},
        {"product_type": "MPL_ORBSCT", "collection_name": "s1-mpl_orbsct"},
        # S2_TILES (SLC__1_NRB__1 pipeline)
        {"product_type": "GIP_TILPAR", "collection_name": "s2-gip_tilpar"},
    ],
    "generated_product_to_collection_identifier": [
        {"name": "cslcs", "product_type": "S01SIWCSL", "collection_name": "s01siwcsl"},
        {"name": "cslcs", "product_type": "S01SEWCSL", "collection_name": "s01sewcsl"},
        {"name": "cslcs", "product_type": "S01SSMCSL", "collection_name": "s01ssmcsl"},
        {"name": "gslcs", "product_type": "S01SIWGSL", "collection_name": "s01siwgsl"},
        {"name": "gslcs", "product_type": "S01SEWGSL", "collection_name": "s01sewgsl"},
        {"name": "gslcs", "product_type": "S01SSMGSL", "collection_name": "s01ssmgsl"},
        {"name": "nrb", "product_type": "S01SIWNRB", "collection_name": "s01siwnrb"},
        {"name": "nrb", "product_type": "S01SEWNRB", "collection_name": "s01sewnrb"},
        {"name": "nrb", "product_type": "S01SSMNRB", "collection_name": "s01ssmnrb"},
    ],
}

# Create the Prefect variable if it doesn't exist, else keep it
s1ard_settings = await Variable.get(S1ARD_SETTINGS_VARIABLE)
if not s1ard_settings:
    print(f"Create the {S1ARD_SETTINGS_VARIABLE!r} Prefect variable")
    await Variable.set(S1ARD_SETTINGS_VARIABLE, DEFAULT_S1ARD_SETTINGS, tags=["s1", "l1", "ard"])
    s1ard_settings = DEFAULT_S1ARD_SETTINGS

# The processor runs in the Dask cluster of the settings
if s1ard_settings.get("dask_cluster_name") != cluster_info_s1ard.cluster_label:
    print(
        f"⚠️ The processor will run in the {s1ard_settings.get('dask_cluster_name')!r} Dask cluster of the "
        f"{S1ARD_SETTINGS_VARIABLE!r} Prefect variable, not in {cluster_info_s1ard.cluster_label!r}")

# Check that the flow searches the SLC collection
input_collections = s1ard_settings.get("input_collections", [])
if SLC_COLLECTION not in input_collections:
    raise RuntimeError(
        f"The 'input_collections' of the {S1ARD_SETTINGS_VARIABLE!r} Prefect variable must contain "
        f"{SLC_COLLECTION!r}, found: {input_collections}")

# Check that the flow doesn't search the SLC in the legacy SAFE collections: the processor cannot read them
legacy_collections = [collection for collection in input_collections if collection.startswith("s1-l1-")]
if legacy_collections:
    raise RuntimeError(
        f"The 'input_collections' of the {S1ARD_SETTINGS_VARIABLE!r} Prefect variable must not contain the "
        f"legacy SAFE SLC collections, remove them: {legacy_collections}")
print(f"S1 ARD settings: {json.dumps(s1ard_settings, indent=2)}")

In [ ]:
# Prefect deployment of the S1 ARD reference flow
# NOTE: the flow reads the EarthDataHub API key, used to read the Copernicus DEM, from the 'EDH_API_KEY' env var
# of the 'env-vars-<owner>' (or 'env-vars') Prefect secret block of the flow owner, i.e. 'env-vars-copernicus'.
S1ARD_REFERENCE_DEPLOYMENT = "process-s1-ard-reference/Process S1 ARD Reference"

params = {
    "slc": REFERENCE_SLC,
    "env_args": {
        "owner_id": OWNER,
        "called_by": "notebooks/demos/s1_ard_processing.ipynb",
    },
}

# Start the flow run now and return immediately (timeout=0) without waiting for it
reference_run = await arun_deployment(
    name=S1ARD_REFERENCE_DEPLOYMENT,
    parameters=params,
    flow_run_name=f"s1-ard-reference-{REFERENCE_SLC}",
    timeout=0,
)
print(f"S1 ARD reference flow run: {os.environ['RSPY_PREFECT_URL']}/runs/flow-run/{reference_run.id}")
print("The path of the REFERENCE_DB is written in the logs of the flow.")

## 9. Check the reference run

Run this cell again to update the state of the reference run.

In [ ]:
async with get_client() as client:
    flow_run = await client.read_flow_run(reference_run.id)
    print(f"{flow_run.name}: {flow_run.state_name}")